In [30]:
import numpy as np
import pandas as pd

In [31]:
df = pd.read_csv('C:\Insurance Predictor\dataset\insurance.csv')

<>:1: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<>:1: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
C:\Users\DELL\AppData\Local\Temp\ipykernel_15112\1252379257.py:1: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
  df = pd.read_csv('C:\Insurance Predictor\dataset\insurance.csv')


In [32]:
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur,retired,High
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore,retired,High


In [33]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
28,38,101.2,1.79,11.630000,False,Mumbai,unemployed,Low
7,31,105.7,1.78,10.865821,True,Delhi,government_job,Medium
36,61,58.4,1.64,0.530000,False,Hyderabad,retired,Medium
90,59,54.0,1.60,21.070000,False,Mumbai,business_owner,Low
71,38,54.1,1.81,20.250000,False,Chandigarh,unemployed,Low


In [34]:
df['occupation'].unique()

<ArrowStringArray>
[       'retired',     'freelancer',        'student', 'government_job',
 'business_owner',     'unemployed',    'private_job']
Length: 7, dtype: str

In [35]:
df_feat = df.copy()

In [36]:
df_feat['bmi'] = df_feat['weight'] / (df_feat['height'] ** 2)

In [37]:
def age_group(age):
    if age < 25:
        return 'young'
    elif age < 45:
        return 'adult'
    elif age < 60:
        return 'middle aged'
    return 'senior'

In [38]:
df_feat['age_group'] = df_feat['age'].apply(age_group)

In [39]:
def lifestyle_risk(row):
    if row['smoker'] and row['bmi'] > 30:
        return 'High'
    elif row['smoker'] and row['bmi'] > 27:
        return 'Medium'
    else:
        return 'Low'

In [40]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis = 1)

In [41]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [42]:
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [43]:
df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [44]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
59,1.13,retired,35.835044,senior,Low,2,High
68,0.68,student,22.963196,young,Low,2,Low
55,24.93,unemployed,25.293194,middle aged,Low,1,Low
77,0.61,retired,37.818734,senior,High,1,High
56,2.86,student,42.414152,young,High,1,Medium


In [45]:
df_feat.columns

Index(['age', 'weight', 'height', 'income_lpa', 'smoker', 'city', 'occupation',
       'insurance_premium_category', 'bmi', 'age_group', 'lifestyle_risk',
       'city_tier'],
      dtype='str')

In [46]:
X = df_feat.drop(columns='insurance_premium_category')
y = df_feat['insurance_premium_category']

In [47]:
X.columns

Index(['age', 'weight', 'height', 'income_lpa', 'smoker', 'city', 'occupation',
       'bmi', 'age_group', 'lifestyle_risk', 'city_tier'],
      dtype='str')

In [48]:
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [49]:
categorical_features

['age_group', 'lifestyle_risk', 'occupation', 'city_tier']

In [50]:
numeric_features

['bmi', 'income_lpa']

In [51]:
from keyword import softkwlist

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(), categorical_features),
        ('num','passthrough', numeric_features)
    ]
)

model1 = RandomForestClassifier(random_state=42)
model2 = DecisionTreeClassifier(max_depth= 4)
model3 = LogisticRegression()
model4 = KNeighborsClassifier(n_neighbors= 7)

ensemble = VotingClassifier(
    estimators=[
        ('rf', model1),
        ('dt', model2),
        ('lor', model3),
        ('knn', model4)
    ],
    voting= 'soft'
)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ensemble)
])

In [52]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size= 0.2, random_state=1)
pipeline.fit(X_train,y_train)

c:\Insurance Predictor\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['age','weight','height',...,'age_group','lifestyle_risk','city_tier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [53]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.75

In [54]:
X_test.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,bmi,age_group,lifestyle_risk,city_tier
52,18,116.7,1.57,2.96,False,Jalandhar,student,47.344720,young,Low,2
31,39,51.1,1.83,11.77,True,Lucknow,private_job,15.258742,adult,Low,2
93,23,79.4,1.85,1.28,False,Indore,student,23.199416,young,Low,2
17,65,90.1,1.70,2.23,False,Delhi,retired,31.176471,senior,Low,1
80,56,95.8,1.67,50.00,False,Jalandhar,unemployed,34.350461,middle aged,Low,2


In [55]:
import pickle

pickle_model_path = "C:\Insurance Predictor\models\model.pkl"
with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline, f)

<>:3: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<>:3: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
C:\Users\DELL\AppData\Local\Temp\ipykernel_15112\1787713095.py:3: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
  pickle_model_path = "C:\Insurance Predictor\models\model.pkl"
